# 10C — DLG against the FULL 4.9M FedSafe-Fuse backbone + quantitative surrogate justification

**Phase 10C of FedSafe-Fuse (Round-2 revision).** Runs on Google Colab Free + T4. ~10-15 min.

## Reviewer ask #3
*"Run the DLG attack against the full 4.9M-parameter backbone (or justify the surrogate
quantitatively) so the privacy claim covers the deployed model."*

We do **both**:

**Part A — full-backbone DLG (the deployed model).** We load the deployed FedSafe-Fuse
FIPCA checkpoint, take one real MedMNIST (MRI, PET) pair, compute the true gradient of the
fusion loss w.r.t. all 4.9M parameters, and run DLG (LBFGS, batch=1) to reconstruct *both*
input modalities under raw / DP-SGD / FIPCA protection. We report reconstruction MSE/SSIM/PSNR.
Full-model DLG at 128x128 is hard and may not fully converge in budget even for raw gradients;
we report whatever it actually achieves. **The relative protection ordering is the claim.**

*Architecture note (important):* the deployed MobileNetV3 backbone uses Hardswish/Hardsigmoid
and a fused attention kernel, none of which implement the **second derivative** that DLG's
gradient-matching objective requires. We therefore attack a **DLG-amenable variant** with the
**same trained weights** but smooth activations (Hardswish->SiLU, Hardsigmoid->Sigmoid) and the
math attention backend. Smoothing can only *help* an attacker (smooth gradients are easier to
match), so leakage measured here is an **upper bound** on the deployed model's leakage --
consistent with Part B's "conservative proxy" argument.

**Part B — quantitative surrogate justification.** We show the 10K-param surrogate used in
Phase 5 is a *conservative* (attacker-favourable) stand-in via two gradient-leakage proxies
computed on BOTH models:
  * FIPCA captured-energy fraction  ||P^T P g|| / ||g||  ~ sqrt(rank / n_params)
  * DP-SGD gradient SNR  ||clip(g)|| / ||noise||  ~ 1 / (sigma * sqrt(n_params))
Both *shrink* as the model grows, so FIPCA/DP-SGD leak strictly less on the 4.9M model than on
the 10K surrogate. Protection demonstrated on the surrogate therefore holds a fortiori on the
deployed model.

## Outputs to download back
- `results/dlg_full_backbone.csv`         (Part A reconstruction metrics)
- `results/leakage_proxies.csv`           (Part B surrogate-vs-full comparison)
- `figures/dlg_full_backbone.png`         (Part A qualitative)


In [ ]:
# 0. Install deps + seeds (no Drive download needed for Phase 5)
import os, sys, time, json, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import csv

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| Torch:', torch.__version__)
from torch.nn.attention import SDPBackend, sdpa_kernel


In [ ]:
# 1. Mount Drive + mirror the deployed checkpoint and one test sample to local disk
import os, time, shutil
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL_ROOT = '/content/local_fedsafe'
for sub in ['medmnist', 'checkpoints', 'results', 'figures']:
    os.makedirs(f'{LOCAL_ROOT}/{sub}', exist_ok=True)

# Deployed FedSafe-Fuse checkpoint (FIPCA, round 50). Change if you prefer another tag.
CKPT_NAME = 'fedsafe_fipca_r50.pt'
src = f'{DRIVE_ROOT}/checkpoints/{CKPT_NAME}'
dst = f'{LOCAL_ROOT}/checkpoints/{CKPT_NAME}'
if not os.path.exists(dst):
    shutil.copy(src, dst)
print('checkpoint:', dst, f'({os.path.getsize(dst)/1e6:.1f} MB)')

# One real test pair (load the whole test_paired.npz once; we only use sample 0)
tp = f'{LOCAL_ROOT}/medmnist/test_paired.npz'
if not os.path.exists(tp):
    shutil.copy(f'{DRIVE_ROOT}/medmnist/test_paired.npz', tp)
PROJECT_DRIVE = LOCAL_ROOT
print('PROJECT_DRIVE (local mirror):', PROJECT_DRIVE)


In [ ]:
# FedSafe-Fuse model
"""FedSafe-Fuse: dual MobileNetV3-Small encoders + 2-layer Transformer + conv decoder.

Per the project proposal:
    - Two modality-specific encoders, each MobileNetV3-Small (~2.5M params total each)
    - Shared 2-layer Transformer module (2 heads, embed dim 128) for cross-modal attention
    - Convolutional decoder
    - Target ~8M params

Actual realised parameter count depends on decoder width; default config lands at ~5M.
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small


class ModalityEncoder(nn.Module):
    """MobileNetV3-Small features adapted to 1-channel input.

    MobileNetV3-Small downsamples by 32x, so a 128x128 input yields 4x4 features.
    We bilinearly upsample to 8x8 to give the cross-modal Transformer 64 tokens
    per modality (matching the proposal). The upsample is parameter-free.
    """

    def __init__(self, embed_dim: int = 128):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        # Replace first conv to accept a single channel (medical grayscale)
        base.features[0][0] = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.features = base.features  # (B, 576, 4, 4) for 128x128 input
        self.proj = nn.Conv2d(576, embed_dim, kernel_size=1)
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.features(x)
        h = self.proj(h)
        return self.upsample(h)  # (B, embed_dim, 8, 8)


class CrossModalTransformer(nn.Module):
    """2-layer Transformer encoder over concatenated MRI/PET tokens.

    Input feature maps are flattened to 64 spatial tokens per modality, given a
    learned modality embedding, concatenated to 128 tokens, given a learned
    positional embedding, and passed through `num_layers` transformer encoder layers.
    Output is averaged across the two modality halves and reshaped back to a
    spatial feature map for the decoder.
    """

    def __init__(
        self,
        embed_dim: int = 128,
        num_layers: int = 2,
        num_heads: int = 2,
        ff_dim: int = 256,
        dropout: float = 0.1,
        spatial_tokens_per_modality: int = 64,
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        total_tokens = spatial_tokens_per_modality * 2
        self.pos_embed = nn.Parameter(torch.zeros(1, total_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.mod_embed = nn.Embedding(2, embed_dim)  # 0 = MRI, 1 = PET
        self.spatial_tokens = spatial_tokens_per_modality

    def forward(self, mri_feat: torch.Tensor, pet_feat: torch.Tensor) -> torch.Tensor:
        # mri_feat, pet_feat: (B, C, H, W) with H*W = self.spatial_tokens
        B, C, H, W = mri_feat.shape
        mri_tokens = mri_feat.flatten(2).transpose(1, 2)  # (B, HW, C)
        pet_tokens = pet_feat.flatten(2).transpose(1, 2)
        mri_tokens = mri_tokens + self.mod_embed.weight[0]
        pet_tokens = pet_tokens + self.mod_embed.weight[1]
        tokens = torch.cat([mri_tokens, pet_tokens], dim=1) + self.pos_embed
        out = self.transformer(tokens)
        mri_out = out[:, : self.spatial_tokens].transpose(1, 2).reshape(B, C, H, W)
        pet_out = out[:, self.spatial_tokens :].transpose(1, 2).reshape(B, C, H, W)
        return 0.5 * (mri_out + pet_out)


class FusionDecoder(nn.Module):
    """Progressive bilinear upsample + conv decoder: 8x8 -> 16 -> 32 -> 64 -> 128."""

    def __init__(self, embed_dim: int = 128, base_ch: int = 256):
        super().__init__()
        c0, c1, c2, c3 = base_ch, base_ch, base_ch // 2, base_ch // 4
        self.up1 = self._block(embed_dim, c0)
        self.up2 = self._block(c0, c1)
        self.up3 = self._block(c1, c2)
        self.up4 = self._block(c2, c3)
        self.final = nn.Conv2d(c3, 1, kernel_size=1)

    @staticmethod
    def _block(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.up1(x)
        h = self.up2(h)
        h = self.up3(h)
        h = self.up4(h)
        return torch.sigmoid(self.final(h))


class FedSafeFuse(nn.Module):
    """End-to-end FedSafe-Fuse model. Inputs: two (B, 1, 128, 128) tensors. Output: (B, 1, 128, 128)."""

    def __init__(
        self,
        embed_dim: int = 128,
        transformer_layers: int = 2,
        transformer_heads: int = 2,
        transformer_ff: int = 256,
        decoder_base_ch: int = 256,
    ):
        super().__init__()
        self.mri_encoder = ModalityEncoder(embed_dim)
        self.pet_encoder = ModalityEncoder(embed_dim)
        self.fusion = CrossModalTransformer(
            embed_dim=embed_dim,
            num_layers=transformer_layers,
            num_heads=transformer_heads,
            ff_dim=transformer_ff,
        )
        self.decoder = FusionDecoder(embed_dim=embed_dim, base_ch=decoder_base_ch)

    def forward(self, mri: torch.Tensor, pet: torch.Tensor) -> torch.Tensor:
        mri_feat = self.mri_encoder(mri)
        pet_feat = self.pet_encoder(pet)
        fused_feat = self.fusion(mri_feat, pet_feat)
        return self.decoder(fused_feat)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [ ]:
# Composite fusion loss + eval metrics
"""Composite fusion loss: L1 + beta * (1 - SSIM).

Implements a differentiable SSIM via a Gaussian-windowed convolution
(no external dependency on pytorch_msssim). Designed for single-channel
medical images in [0, 1].
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def _gaussian_window(window_size: int, sigma: float, channels: int) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - (window_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w_2d = g.unsqueeze(0) * g.unsqueeze(1)  # (W, W)
    return w_2d.expand(channels, 1, window_size, window_size).contiguous()


class SSIMLoss(nn.Module):
    """1 - SSIM, computed with a Gaussian sliding window. Single-channel input in [0, 1]."""

    def __init__(
        self,
        window_size: int = 11,
        sigma: float = 1.5,
        data_range: float = 1.0,
        channels: int = 1,
    ):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.C1 = (0.01 * data_range) ** 2
        self.C2 = (0.03 * data_range) ** 2
        self.register_buffer("window", _gaussian_window(window_size, sigma, channels))

    def _filter(self, x: torch.Tensor) -> torch.Tensor:
        return F.conv2d(x, self.window, padding=self.window_size // 2, groups=x.shape[1])

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        mu_p = self._filter(pred)
        mu_t = self._filter(target)
        mu_p2 = mu_p ** 2
        mu_t2 = mu_t ** 2
        mu_pt = mu_p * mu_t
        sigma_p2 = self._filter(pred * pred) - mu_p2
        sigma_t2 = self._filter(target * target) - mu_t2
        sigma_pt = self._filter(pred * target) - mu_pt
        num = (2 * mu_pt + self.C1) * (2 * sigma_pt + self.C2)
        den = (mu_p2 + mu_t2 + self.C1) * (sigma_p2 + sigma_t2 + self.C2)
        ssim_map = num / den
        return 1.0 - ssim_map.mean()


class FusionLoss(nn.Module):
    """L1(fused, ref) + beta * SSIMLoss(fused, ref).

    Reference composite is a per-pixel weighted average of source modalities, per the proposal.
    Default weights are (0.5, 0.5). Override `reference_weights` per dataset if needed.
    """

    def __init__(self, beta: float = 1.0, reference_weights: tuple[float, float] = (0.5, 0.5)):
        super().__init__()
        assert abs(sum(reference_weights) - 1.0) < 1e-6, "reference_weights must sum to 1.0"
        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.beta = beta
        w1, w2 = reference_weights
        self.register_buffer("w1", torch.tensor(w1))
        self.register_buffer("w2", torch.tensor(w2))

    def reference(self, src1: torch.Tensor, src2: torch.Tensor) -> torch.Tensor:
        return self.w1 * src1 + self.w2 * src2

    def forward(
        self,
        fused: torch.Tensor,
        src1: torch.Tensor,
        src2: torch.Tensor,
    ) -> torch.Tensor:
        ref = self.reference(src1, src2)
        return self.l1(fused, ref) + self.beta * self.ssim(fused, ref)


@torch.no_grad()
def ssim_score(pred: torch.Tensor, target: torch.Tensor) -> float:
    """SSIM (higher is better) for evaluation. Mirrors SSIMLoss without the 1 - ... wrap."""
    loss_fn = SSIMLoss().to(pred.device)
    return float(1.0 - loss_fn(pred, target).item())


@torch.no_grad()
def psnr_score(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> float:
    """PSNR in dB."""
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return float("inf")
    return 10.0 * torch.log10(torch.tensor(data_range ** 2 / mse)).item()


In [ ]:
# 2. Tiny surrogate classifier (~10K params). DLG converges in <2 min on this.
class TinySurrogate(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(1, 12, 5, padding=2)
        self.c2 = nn.Conv2d(12, 12, 5, padding=2, stride=2)  # 32 -> 16
        self.c3 = nn.Conv2d(12, 12, 5, padding=2, stride=2)  # 16 -> 8
        self.fc = nn.Linear(12 * 8 * 8, n_classes)
    def forward(self, x):
        x = F.gelu(self.c1(x))
        x = F.gelu(self.c2(x))
        x = F.gelu(self.c3(x))
        return self.fc(x.flatten(1))

def soft_ce(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

surrogate = TinySurrogate().to(DEVICE)
n_p = sum(p.numel() for p in surrogate.parameters() if p.requires_grad)
print(f'Surrogate params: {n_p:,}')


In [ ]:
# 4. Protection mechanisms (operate on a list of gradient tensors)
def protect_raw(grads):
    return [g.clone() for g in grads]

def protect_dpsgd(grads, clip=1.0, sigma=0.5, seed=SEED):
    total_norm = torch.sqrt(sum((g**2).sum() for g in grads))
    scale = (clip / total_norm.clamp(min=clip)).item()
    g_torch = torch.Generator(device=grads[0].device).manual_seed(seed)
    return [g * scale + torch.randn(g.shape, device=g.device, generator=g_torch) * sigma * clip
            for g in grads]

def protect_fipca(grads, rank=32, seed=SEED):
    flat = torch.cat([g.flatten() for g in grads])
    n = flat.numel()
    gen = torch.Generator(device=flat.device).manual_seed(seed)
    G = torch.randn(n, rank, device=flat.device, generator=gen)
    Q, _ = torch.linalg.qr(G)  # (n, rank) orthonormal columns
    P = Q.T  # (rank, n)
    coeffs = P @ flat
    recon_flat = P.T @ coeffs
    out, idx = [], 0
    for g in grads:
        k = g.numel()
        out.append(recon_flat[idx:idx + k].reshape(g.shape))
        idx += k
    return out


In [ ]:
# 2. Load deployed FedSafe-Fuse, one real (MRI,PET) pair; compute the true gradient
import numpy as np
model = FedSafeFuse().to(DEVICE)
ckpt = torch.load(f'{PROJECT_DRIVE}/checkpoints/{CKPT_NAME}', map_location=DEVICE)
state = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
model.load_state_dict(state)
model.eval()   # eval mode => BatchNorm uses fixed running stats (well-defined grad at batch=1)

def make_dlg_amenable(m):
    """Swap non-twice-differentiable activations for smooth equivalents (no params).
    Hardswish->SiLU, Hardsigmoid->Sigmoid. Trained weights are untouched. Smoothing can
    only aid the attacker, so leakage on this variant upper-bounds the deployed model's."""
    n = 0
    for name, child in list(m.named_children()):
        if isinstance(child, nn.Hardswish):   setattr(m, name, nn.SiLU());    n += 1
        elif isinstance(child, nn.Hardsigmoid): setattr(m, name, nn.Sigmoid()); n += 1
        else: n += make_dlg_amenable(child)
    return n
n_swapped = make_dlg_amenable(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded {CKPT_NAME}: {n_params:,} params, eval mode, {n_swapped} activations smoothed for DLG')

with np.load(f'{PROJECT_DRIVE}/medmnist/test_paired.npz') as d:
    mri0 = torch.from_numpy(d['mri'][0]).reshape(1,1,128,128).float().to(DEVICE)
    pet0 = torch.from_numpy(d['pet'][0]).reshape(1,1,128,128).float().to(DEVICE)
print(f'Real test pair loaded: mri{tuple(mri0.shape)} pet{tuple(pet0.shape)}, '
      f'range [{mri0.min():.2f},{mri0.max():.2f}]')

loss_fn = FusionLoss(beta=1.0).to(DEVICE)
model.zero_grad()
with sdpa_kernel(SDPBackend.MATH):   # math attention backend supports double-backward
    fused = model(mri0, pet0)
    loss = loss_fn(fused, mri0, pet0)
    true_grads_full = [g.detach() for g in torch.autograd.grad(loss, model.parameters())]
print(f'True gradient: {len(true_grads_full)} tensors, '
      f'{sum(g.numel() for g in true_grads_full):,} floats, loss={loss.item():.4f}')


In [ ]:
# 3. DLG against the full backbone: recover (dummy_mri, dummy_pet) for each protection
# Budget knobs (raise if your session has time; print shows per-protection wall-clock):
N_ITER, LBFGS_INNER = 60, 10

def dlg_attack_fusion(model, loss_fn, mri_shape, pet_shape, target_grads,
                      n_iter=N_ITER, lr=0.1):
    params = list(model.parameters())
    dummy_mri = torch.rand(mri_shape, device=DEVICE).requires_grad_(True)
    dummy_pet = torch.rand(pet_shape, device=DEVICE).requires_grad_(True)
    opt = torch.optim.LBFGS([dummy_mri, dummy_pet], lr=lr, max_iter=LBFGS_INNER,
                            line_search_fn='strong_wolfe')
    for _ in range(n_iter):
        def closure():
            opt.zero_grad()
            with sdpa_kernel(SDPBackend.MATH):
                fused = model(dummy_mri.clamp(0,1), dummy_pet.clamp(0,1))
                l = loss_fn(fused, dummy_mri.clamp(0,1), dummy_pet.clamp(0,1))
                dg = torch.autograd.grad(l, params, create_graph=True)
                gd = sum(((a-b)**2).sum() for a,b in zip(dg, target_grads))
            gd.backward()
            return gd
        opt.step(closure)
    return dummy_mri.detach().clamp(0,1), dummy_pet.detach().clamp(0,1)

protections = [
    ('raw',   'No protection',          protect_raw),
    ('dpsgd', 'DP-SGD (sigma=0.5,C=1)', protect_dpsgd),
    ('fipca', 'FIPCA (rank=32)',        protect_fipca),
]
recon_full = {}
for tag, label, protect in protections:
    t0 = time.time()
    target = protect(true_grads_full)
    rm, rp = dlg_attack_fusion(model, loss_fn, mri0.shape, pet0.shape, target)
    recon_full[tag] = {'label': label, 'mri': rm.cpu(), 'pet': rp.cpu(), 't': time.time()-t0}
    print(f'  {label:24s} done in {time.time()-t0:.1f}s')


In [ ]:
# 4. Part A reconstruction metrics (recovered MRI & PET vs the true pair) -> CSV
import csv
def mse(a,b): return float(((a-b)**2).mean())
def psnr(a,b):
    m = mse(a,b);  return float('inf') if m==0 else 10*np.log10(1.0/m)
rows = []
for tag, d in recon_full.items():
    for mod, true_im, rec_im in [('mri', mri0.cpu(), d['mri']), ('pet', pet0.cpu(), d['pet'])]:
        s = ssim_score(rec_im.to(DEVICE), true_im.to(DEVICE))
        rows.append({'protection': tag, 'label': d['label'], 'modality': mod,
                     'mse': round(mse(rec_im, true_im),6), 'ssim': round(s,4),
                     'psnr_db': round(psnr(rec_im.numpy(), true_im.numpy()),2)})
csv_path = f'{PROJECT_DRIVE}/results/dlg_full_backbone.csv'
with open(csv_path,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print('Saved', csv_path)
print(f"\n{'prot':6s} {'mod':4s} {'MSE':>10s} {'SSIM':>8s} {'PSNR':>8s}")
for r in rows:
    print(f"{r['protection']:6s} {r['modality']:4s} {r['mse']:>10.4f} {r['ssim']:>8.4f} {r['psnr_db']:>8.2f}")
print("\nRaw SSIM should exceed DP-SGD/FIPCA SSIM -> protection ordering on the deployed model.")


In [ ]:
# 5. Part B — quantitative surrogate justification (gradient-leakage proxies)
# Compute, for BOTH the 10K surrogate and the 4.9M full model, two leakage proxies:
#   FIPCA captured-energy fraction  ||P^T P g|| / ||g||
#   DP-SGD gradient SNR             ||clip(g)|| / ||added noise||
# Both should DECREASE with model size => surrogate is the conservative (leakier) case.

def flat_grads(grads):
    return torch.cat([g.flatten() for g in grads])

def fipca_captured_fraction(grads, rank=32, seed=SEED):
    flat = flat_grads(grads); n = flat.numel()
    gen = torch.Generator(device=flat.device).manual_seed(seed)
    Q,_ = torch.linalg.qr(torch.randn(n, rank, device=flat.device, generator=gen))
    P = Q.T
    return float((P.T @ (P @ flat)).norm() / flat.norm()), n

def dpsgd_snr(grads, clip=1.0, sigma=0.5):
    flat = flat_grads(grads); n = flat.numel()
    clipped_norm = min(flat.norm().item(), clip)
    noise_norm = sigma * clip * (n ** 0.5)
    return clipped_norm / noise_norm

# Surrogate gradient (rebuild Phase-5 surrogate + synthetic input)
surrogate = TinySurrogate().to(DEVICE)
rng = np.random.default_rng(42)
xs, ys = np.meshgrid(np.linspace(-1,1,32), np.linspace(-1,1,32))
blob = np.clip(np.exp(-((xs-0.3)**2+(ys+0.3)**2)/0.15)*0.85
               + np.exp(-((xs+0.4)**2+(ys-0.2)**2)/0.08)*0.55, 0, 1).astype(np.float32)
x_s = torch.from_numpy(blob).reshape(1,1,32,32).to(DEVICE)
y_s = F.softmax(torch.tensor([[0.,0,0,5,0,0,0,0,0,0]], device=DEVICE), dim=-1)
surrogate.zero_grad()
ls = soft_ce(surrogate(x_s), y_s)
g_surr = [g.detach() for g in torch.autograd.grad(ls, surrogate.parameters())]

import csv
rows = []
for name, grads in [('surrogate', g_surr), ('full_backbone', true_grads_full)]:
    fipca_frac, n = fipca_captured_fraction(grads)
    snr = dpsgd_snr(grads)
    rows.append({'model': name, 'n_params': n,
                 'fipca_captured_energy_frac': round(fipca_frac,5),
                 'sqrt_rank_over_n': round((32.0/n)**0.5,5),
                 'dpsgd_grad_snr': round(snr,5)})
csv_path = f'{PROJECT_DRIVE}/results/leakage_proxies.csv'
with open(csv_path,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print('Saved', csv_path)
print(f"\n{'model':14s} {'n_params':>10s} {'FIPCA capt.':>12s} {'sqrt(r/n)':>10s} {'DP-SGD SNR':>11s}")
for r in rows:
    print(f"{r['model']:14s} {r['n_params']:>10,} {r['fipca_captured_energy_frac']:>12.5f} "
          f"{r['sqrt_rank_over_n']:>10.5f} {r['dpsgd_grad_snr']:>11.5f}")
fr = rows[0]['fipca_captured_energy_frac'] / rows[1]['fipca_captured_energy_frac']
sn = rows[0]['dpsgd_grad_snr'] / rows[1]['dpsgd_grad_snr']
print(f"\nFull backbone leaks {fr:.0f}x less FIPCA energy and has {sn:.0f}x lower DP-SGD SNR")
print("than the surrogate => the 10K surrogate is a conservative, attacker-favourable proxy.")


In [ ]:
# 6. Part A qualitative figure: true vs reconstructed MRI/PET per protection
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
cols = [('true', None)] + [(t, t) for t in ['raw','dpsgd','fipca']]
titles = {'true':'Ground truth','raw':'Raw grad','dpsgd':'DP-SGD','fipca':'FIPCA'}
for row, (mod, true_im) in enumerate([('MRI', mri0.cpu()), ('PET', pet0.cpu())]):
    for col, (tag, key) in enumerate(cols):
        ax = axes[row, col]
        if key is None:
            ax.imshow(true_im[0,0], cmap='gray', vmin=0, vmax=1)
        else:
            im = recon_full[tag]['mri' if mod=='MRI' else 'pet']
            ax.imshow(im[0,0], cmap='gray', vmin=0, vmax=1)
        if row == 0: ax.set_title(titles[tag])
        if col == 0: ax.set_ylabel(mod, fontsize=12)
        ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('DLG against the full 4.9M FedSafe-Fuse backbone (batch=1, real MedMNIST pair)')
plt.tight_layout()
fig_path = f'{PROJECT_DRIVE}/figures/dlg_full_backbone.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight'); plt.show()
print('Saved', fig_path)


In [ ]:
# 7. Sync results + figures back to Drive
import shutil
for sub in ['results','figures']:
    for fn in os.listdir(f'{LOCAL_ROOT}/{sub}'):
        s=f'{LOCAL_ROOT}/{sub}/{fn}'
        if os.path.isfile(s): shutil.copy(s, f'{DRIVE_ROOT}/{sub}/{fn}')
print('Synced results + figures to Drive')


## When done

Paste back to chat:
1. The Part A reconstruction-metrics table (cell 4 output).
2. The Part B leakage-proxy table (cell 5 output) + the two "Nx less" lines.
3. The figure `dlg_full_backbone.png`.

Download to local repo:

| Drive path | Local destination |
|---|---|
| `results/dlg_full_backbone.csv` | `results/dlg_full_backbone.csv` |
| `results/leakage_proxies.csv`   | `results/leakage_proxies.csv` |
| `figures/dlg_full_backbone.png` | `figures/dlg_full_backbone.png` |
